In [1]:
from geopy.geocoders import Nominatim
import pandas as pd
import geopandas as gpd
import h3pandas
import time
from shapely.geometry import Point

In [2]:
geolocator = Nominatim(user_agent="rais-geocoding")

In [13]:
YEAR = "2013"

jobs = pd.read_csv(f"../data/address/reallyNotFound__{YEAR}.csv", sep=';')
jobs = jobs.groupby(["nome_logradouro", "NUM_ENDERECO", "bairro"]).agg({"estoque": "sum", "_index": "first"}).reset_index()

In [14]:
jobs.head(2)

,nome_logradouro,NUM_ENDERECO,bairro,estoque,_index
0,341,417.0,SANTA EFIGENIA,2,10701
1,AC AV JOSE DE SOUZA CAMPOS,753.0,CAMBUI,7,186596


In [ ]:
_count = 0
notFound = 0
found = 0

locations = []

In [17]:
for i, row in jobs.iloc[1831:].iterrows():
    
    try:
        address = str(row.nome_logradouro) + ", " + str(int(row.NUM_ENDERECO)) + ", " + "São Paulo, São Paulo, Brasil"
        location = geolocator.geocode(address)
        locations.append({"latitude": location.latitude, "longitude": location.longitude, "_index": row["_index"]})
        found += 1
    except:
        notFound += 1
        if notFound%30 == 0: 
            print("****")
            print(_count, 'not found... while found', found)
            print()

    _count += 1
    if _count%120 == 0: print(_count, 'has gone...', found, 'worked')
    #if _count==50: break
    #print(_count, 'keep waiting...')
    #break

    time.sleep(0.8)

#bench: 270 completed... while found 181


1920 has gone... 1474 worked
****
1951 not found... while found 1502

2040 has gone... 1577 worked
****
2093 not found... while found 1614

2160 has gone... 1651 worked
****
2160 not found... while found 1651

2280 has gone... 1753 worked
****
2317 not found... while found 1778

2400 has gone... 1844 worked
2520 has gone... 1961 worked
****
2537 not found... while found 1968

2640 has gone... 2049 worked
****
2688 not found... while found 2089

2760 has gone... 2149 worked
****
2848 not found... while found 2219

2880 has gone... 2245 worked
3000 has gone... 2349 worked
****
3034 not found... while found 2375

3120 has gone... 2433 worked
****
3128 not found... while found 2439

3240 has gone... 2530 worked
3360 has gone... 2644 worked
****
3396 not found... while found 2677

3480 has gone... 2740 worked
****
3585 not found... while found 2836

3600 has gone... 2848 worked
3720 has gone... 2950 worked
****
3744 not found... while found 2965

****
3806 not found... while found 2997

384

In [18]:
_count

8877

In [19]:
newLocDf = pd.DataFrame(locations)

In [20]:
foundJobs = jobs.merge(newLocDf, on="_index")
len(foundJobs), foundJobs.estoque.sum()

(6834, 245240)

In [22]:
len(foundJobs), len(jobs), foundJobs.estoque.sum()

(6834, 8878, 245240)

In [23]:
reallyNotFound2 = jobs[~jobs["_index"].isin(foundJobs["_index"])]
reallyNotFound2.to_csv(f"../data/address/reallyNotFound2__{YEAR}.csv", index=False, sep=';')

In [24]:
reallyNotFound2.estoque.sum(), jobs.estoque.sum()

(74854, 320094)

In [25]:
foundJobs['geometry'] = foundJobs.apply(lambda _row: Point(_row.longitude, _row.latitude), axis=1)
foundJobs = gpd.GeoDataFrame(foundJobs)
foundJobs = foundJobs.h3.geo_to_h3(9).reset_index()

foundJobs = foundJobs.rename(columns={"estoque": "T001", "h3_09": 'id_hex'})
print(foundJobs.T001.sum(), jobs.estoque.sum())
print('now building csv:::: almost done!')
#merged.to_file(f"./data/address/origins__{YEAR}.geojson", driver='GeoJSON')
foundJobs.drop(columns=['geometry']).to_csv(f"../data/address/_destinations2__{YEAR}.csv", sep=';', index=False)

245240 320094
now building csv:::: almost done!
